# DirectoryLoader: folder of documents to RAG pipeline

This notebook loads every `.txt` file in a folder, builds a local FAISS index, retrieves the most relevant passages, and generates a grounded answer with Groq.

## 1. Imports and project paths

`DirectoryLoader` is useful when a knowledge base is stored as many files. It automatically creates one LangChain document per file and preserves the file path in document metadata for citations and debugging.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or its notebook folder.')

PROJECT_ROOT = find_project_root()
TEXT_DIRECTORY = PROJECT_ROOT / 'data' / 'text_files'
assert TEXT_DIRECTORY.is_dir(), f'Text directory not found: {TEXT_DIRECTORY}'
print(f'Using directory: {TEXT_DIRECTORY}')

## 2. Load every text file

In [ ]:
loader = DirectoryLoader(
    str(TEXT_DIRECTORY),
    glob='**/*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True,
)
documents = loader.load()
if not documents:
    raise ValueError(f'No .txt files found in {TEXT_DIRECTORY}')

print(f'Loaded {len(documents)} files.')
for document in documents:
    print(document.metadata['source'])

## 3. Chunk, embed, and index documents

The text splitter produces retrieval-sized passages. Local sentence-transformer embeddings capture meaning, while FAISS retrieves the closest passages efficiently.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks, embeddings)
print(f'Indexed {len(chunks)} chunks from {len(documents)} files.')

## 4. Retrieve context and generate an answer

The prompt instructs the LLM to answer only from retrieved passages. Put `GROQ_API_KEY=...` in the project `.env` file before running.

In [ ]:
load_dotenv(PROJECT_ROOT / '.env')
groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    raise RuntimeError('Set GROQ_API_KEY in .env before running the LLM cell.')

question = 'What is machine learning?'
retrieved_docs = vector_store.similarity_search(question, k=3)
context = '\n\n'.join(doc.page_content for doc in retrieved_docs)

prompt = PromptTemplate.from_template(
    'Answer only from the supplied context. If the context is insufficient, say so.\n\n'
    'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
)
llm = ChatGroq(api_key=groq_api_key, model='openai/gpt-oss-20b', temperature=0.1, max_tokens=512)
answer = llm.invoke(prompt.format(context=context, question=question))

print(answer.content)
print('\nRetrieved files:', [doc.metadata.get('source') for doc in retrieved_docs])